# Trabalho Final ALC — Método 2: Esteganografia por Rotação da Matriz $U$

Este notebook implementa um **método alternativo** de esteganografia baseada em SVD, para fins de comparação com o método principal do relatório (inserção de bits na paridade dos valores singulares de $\Sigma$).

Em vez de perturbar $\Sigma$, este método insere a mensagem na matriz ortogonal $U$, por meio de pequenas **rotações de Givens** aplicadas a pares de colunas — de forma que $\Sigma$ permanece **exatamente inalterada**. Essa escolha de design é mais próxima, em espírito, da ideia original de Bergman e Davidson (2005) de esconder informação nos vetores singulares em vez dos valores singulares — embora a técnica exata de codificação (rotação, em vez de inversão de sinal) e de extração (por chave, em vez de recomputação cega da SVD) sejam diferentes das do artigo.

# 1) Importação das bibliotecas necessárias

In [ ]:
from matplotlib.image import imread
import matplotlib.pyplot as plt
import numpy as np
import os
import time

# 2) Funções auxiliares

A conversão de texto para bits usa UTF-8 (em vez de `ord()`/8 bits por caractere), o que lida corretamente com caracteres acentuados — diferença importante em relação ao método de $\Sigma$, que exige mensagens sem acentos.

In [ ]:
def print_matrix(M, name="Matriz", decimals=6):
    with np.printoptions(precision=decimals, suppress=True, linewidth=200, threshold=np.inf):
        print(f"\n{name} ({M.shape[0]} x {M.shape[1]})")
        print(M)


def texto_para_bits(texto):
    dados = texto.encode("utf-8") + b"\0"
    return ''.join(format(byte, '08b') for byte in dados)


def bits_para_texto(bits):
    bytes_mensagem = []
    for i in range(0, len(bits), 8):
        byte = bits[i:i+8]
        if len(byte) < 8:
            break
        valor = int(byte, 2)
        if valor == 0:
            break
        bytes_mensagem.append(valor)
    return bytes(bytes_mensagem).decode("utf-8", errors="replace")

# 3) Digitar a mensagem

In [ ]:
mensagem = input("Digite a mensagem secreta: ")
bits = texto_para_bits(mensagem)
bits_originais = bits  # cópia para avaliação da taxa de erro mais adiante

print("\nMensagem:")
print(mensagem)
print("\nQuantidade de bits:")
print(len(bits))
print("\nBits:")
print(bits)

# 4) Entrada da Imagem

A imagem é convertida para tons de cinza, pois assim podemos trabalhar com uma única matriz numérica, facilitando a aplicação da decomposição em valores singulares.

Coloque a imagem dentro de uma pasta `DATA` (mesma convenção usada no notebook do método de $\Sigma$), e ajuste o nome do arquivo abaixo.

In [ ]:
plt.rcParams['figure.figsize'] = [15, 8]

A = imread(os.path.join("DATA", "Passaro.jpg"))

# Converte para tons de cinza, caso a imagem seja RGB ou RGBA
if A.ndim == 3:
    X = np.mean(A[:, :, :3], axis=2)
else:
    X = A.astype(float)

X = X.astype(float)

plt.imshow(X, cmap='gray')
plt.title("Imagem Original")
plt.axis('off')
plt.show()

print("Dimensão da imagem:", X.shape)

# 5) Decomposição SVD da imagem

$$X = U \Sigma V^T$$

- $U$ — vetores ortonormais associados às direções principais da imagem (é aqui que a mensagem será inserida);
- $\Sigma$ — valores singulares (permanecerão **inalterados** neste método);
- $V^T$ — outra base ortonormal.

Valores singulares praticamente nulos ($<10^{-10}$) são descartados da parte "útil" para a codificação, pois causariam instabilidade numérica na extração (que envolve dividir por $\Sigma$ — ver Seção 8).

In [ ]:
inicio_svd = time.time()
U, s, VT = np.linalg.svd(X, full_matrices=False)
tempo_svd = time.time() - inicio_svd
print(f"Tempo de execução da SVD: {tempo_svd:.4f} s")

cond = np.linalg.cond(X)
print(f"Número de condição κ₂(X) = {cond:.2e}")

tol = 1e-10
k_util = int(np.sum(s > tol))

U_chave = U[:, :k_util]
s_chave = s[:k_util]
VT_chave = VT[:k_util, :]

print("Quantidade total de valores singulares:", len(s))
print("Quantidade útil para codificação:", k_util)

# 6) Parâmetros da codificação

`theta` é o ângulo de rotação usado para codificar cada bit. `inicio_colunas` define a partir de qual coluna de $U$ a mensagem começa a ser inserida — as primeiras colunas (associadas aos maiores valores singulares) são deixadas intactas, seguindo a mesma lógica de proteção usada no método de $\Sigma$ e motivada por Bergman e Davidson (2005): as direções principais concentram a informação estrutural da imagem.

In [ ]:
theta = 0.01
inicio_colunas = 20

capacidade = (k_util - inicio_colunas) // 2

print("Capacidade em bits:", capacidade)
print("Quantidade de bits da mensagem:", len(bits))

if len(bits) > capacidade:
    raise ValueError(
        "Mensagem muito grande para esta imagem usando a matriz U. "
        "Tente uma imagem maior, uma mensagem menor ou diminua inicio_colunas."
    )

# 7) Matriz ortogonal $Q$

Para alterar $U$ sem destruir sua ortonormalidade, construímos uma matriz ortogonal $Q$ formada por pequenas rotações de Givens. Cada bit da mensagem é representado pelo **sentido** da rotação: um sentido para o bit 1, o oposto para o bit 0. Cada bit ocupa um par de colunas exclusivo (sem sobreposição entre bits), então $Q$ é ortogonal por construção (bloco-diagonal de rotações ortogonais + identidade).

In [ ]:
def criar_Q_unitaria(bits, k, theta=0.01, inicio_colunas=20):
    Q = np.eye(k)
    for idx, bit in enumerate(bits):
        i = inicio_colunas + 2 * idx
        j = i + 1
        angulo = theta if bit == "1" else -theta
        c = np.cos(angulo)
        sen = np.sin(angulo)
        Q[np.ix_([i, j], [i, j])] = np.array([[c, -sen], [sen, c]])
    return Q

# 8) Inserir mensagem na matriz $U$

$$U_{\text{estego}} = U Q$$

Como $Q$ é ortogonal, $U_{\text{estego}}$ preserva a estrutura ortonormal de $U$ (verificado numericamente abaixo).

In [ ]:
Q = criar_Q_unitaria(bits=bits, k=k_util, theta=theta, inicio_colunas=inicio_colunas)
U_estego_chave = U_chave @ Q

erro_ortogonalidade = np.linalg.norm(U_estego_chave.T @ U_estego_chave - np.eye(k_util))
print(f"Erro de ortogonalidade de U_estego = {erro_ortogonalidade:.6e}")

# 9) Reconstrução da imagem esteganográfica

$$X_{\text{estego}} = U_{\text{estego}} \, \Sigma \, V^T$$

$\Sigma$ e $V^T$ permanecem inalterados.

In [ ]:
S_chave = np.diag(s_chave)
X_estego = U_estego_chave @ S_chave @ VT_chave

# Componentes fora da parte útil (valores singulares ~0) são somados de volta
if k_util < len(s):
    X_restante = U[:, k_util:] @ np.diag(s[k_util:]) @ VT[k_util:, :]
    X_estego = X_estego + X_restante

valor_max = 1.0 if X.max() <= 1.5 else 255.0
X_estego_vis = np.clip(X_estego, 0, valor_max)

# 10) Verificação: $\Sigma$ permaneceu inalterado?

Diferente do método de $\Sigma$ (onde a perturbação dos valores singulares é o próprio mecanismo de inserção), aqui $\Sigma$ deveria continuar **idêntico** ao original — essa é a principal alegação de vantagem do método (Seção 12 abaixo). Vale checar isso diretamente, recompondo a SVD da imagem estego e comparando os valores singulares.

In [ ]:
_, s_estego_recalculado, _ = np.linalg.svd(X_estego, full_matrices=False)

diff_sigma = np.abs(s_estego_recalculado[:k_util] - s_chave)
print(f"Maior diferença entre Σ original e Σ recalculado: {diff_sigma.max():.3e}")
print(f"Diferença média: {diff_sigma.mean():.3e}")

# 11) Erro de Frobenius, MSE, PSNR e SSIM

Métricas de qualidade da imagem esteganográfica em relação à original — mesmas métricas calculadas no método de $\Sigma$, para permitir comparação direta na tabela de Resultados.

In [ ]:
def calcular_mse(A, B):
    return np.mean((A - B) ** 2)


def calcular_psnr(A, B):
    mse = calcular_mse(A, B)
    if mse == 0:
        return np.inf
    valor_max_local = 1.0 if A.max() <= 1.5 else 255.0
    return 20 * np.log10(valor_max_local / np.sqrt(mse))


erro_frobenius = np.linalg.norm(X - X_estego)
mse = calcular_mse(X, X_estego)
psnr = calcular_psnr(X, X_estego)

print(f"Erro de Frobenius = {erro_frobenius:.6f}")
print(f"MSE = {mse:.10f}")
print(f"PSNR = {psnr:.6f} dB")

try:
    from skimage.metrics import structural_similarity
    ssim = structural_similarity(X, X_estego, data_range=valor_max)
    print(f"SSIM = {ssim:.4f}")
except ImportError:
    print("scikit-image não encontrado; rode 'pip install scikit-image' para o SSIM.")

# 12) Comparação entre a imagem original e a esteganográfica

In [ ]:
plt.figure(figsize=(18, 6))

plt.subplot(131)
plt.imshow(X, cmap='gray')
plt.title("Imagem Original")
plt.axis('off')

plt.subplot(132)
plt.imshow(X_estego_vis, cmap='gray')
plt.title("Imagem Esteganográfica — Alterando U")
plt.axis('off')

plt.subplot(133)
plt.imshow(X_estego - X, cmap='gray')
plt.title("Diferença")
plt.colorbar()

plt.show()

In [ ]:
for r in (5, 50, 100, 200):
    if r > k_util:
        continue
    Xapprox = U_estego_chave[:, :r] @ np.diag(s_chave[:r]) @ VT_chave[:r, :]
    Xapprox_vis = np.clip(Xapprox, 0, valor_max)

    plt.figure(figsize=(8, 4))
    plt.subplot(121)
    plt.imshow(X, cmap='gray')
    plt.title("Imagem Original")
    plt.axis('off')

    plt.subplot(122)
    plt.imshow(Xapprox_vis, cmap='gray')
    plt.title(f"Imagem Esteganográfica U — r = {r}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# 13) Salvar imagem esteganográfica e chave

**Ponto central da comparação com o método de $\Sigma$:** este método **não é cego** — a extração exige uma chave com $U$, $\Sigma$ e $V^T$ originais, pois recalcular a SVD de $X_{\text{estego}}$ diretamente não garante recuperar o mesmo $U$ usado na codificação (ambiguidades de sinal e possíveis reordenações). A célula abaixo também reporta o **tamanho da chave** comparado ao da imagem, para quantificar esse custo.

In [ ]:
np.save("imagem_estego_U.npy", X_estego)
np.savez(
    "chave_svd_U.npz",
    U=U_chave, s=s_chave, VT=VT_chave,
    n_bits=len(bits), theta=theta, inicio_colunas=inicio_colunas
)

tamanho_imagem = os.path.getsize("imagem_estego_U.npy")
tamanho_chave = os.path.getsize("chave_svd_U.npz")

print("Imagem esteganográfica salva como imagem_estego_U.npy"
      f" ({tamanho_imagem/1024:.1f} KB)")
print("Chave de decodificação salva como chave_svd_U.npz"
      f" ({tamanho_chave/1024:.1f} KB)")
print(f"Razão chave/imagem: {tamanho_chave/tamanho_imagem:.2f}x")

# 14) Testes Exploratórios: Robustez (Quantização e JPEG)

Mesmos testes aplicados ao método de $\Sigma$, para comparação direta. Como a informação aqui está no **sinal de uma rotação** em $U$ (não na paridade de um valor numérico), a hipótese é que a robustez a arredondamento possa ser diferente — vale medir, não assumir.

In [ ]:
# Teste de quantização: a chave continua válida se a imagem estego for
# convertida para uint8 (como qualquer imagem real exigiria)?
X_estego_uint8 = np.clip(np.round(X_estego), 0, 255).astype(np.uint8)

from PIL import Image
Image.fromarray(X_estego_uint8).save("estego_U_quantizado.png")
X_recarregada = np.array(Image.open("estego_U_quantizado.png"), dtype=np.float64)

V_original = VT_chave.T
S_inv = np.diag(1 / s_chave)
U_recuperado_quant = X_recarregada[:, :VT_chave.shape[1]] @ V_original @ S_inv
Q_recuperado_quant = U_chave.T @ U_recuperado_quant

bits_quant = ""
for idx in range(len(bits_originais)):
    i = inicio_colunas + 2 * idx
    j = i + 1
    bits_quant += "1" if Q_recuperado_quant[j, i] > 0 else "0"

erros_quant = sum(b1 != b2 for b1, b2 in zip(bits_originais, bits_quant))
print(f"Taxa de erro pós-quantização = {erros_quant/len(bits_originais):.4f} "
      f"({erros_quant}/{len(bits_originais)} bits)")
print("Mensagem pós-quantização:", repr(bits_para_texto(bits_quant)))

In [ ]:
# Teste de robustez a recompressão JPEG
Image.fromarray(X_estego_uint8).save("estego_U_teste.jpg", quality=90)
X_recompactada = np.array(Image.open("estego_U_teste.jpg").convert("L"), dtype=np.float64)

U_recuperado_jpeg = X_recompactada[:, :VT_chave.shape[1]] @ V_original @ S_inv
Q_recuperado_jpeg = U_chave.T @ U_recuperado_jpeg

bits_jpeg = ""
for idx in range(len(bits_originais)):
    i = inicio_colunas + 2 * idx
    j = i + 1
    bits_jpeg += "1" if Q_recuperado_jpeg[j, i] > 0 else "0"

erros_jpeg = sum(b1 != b2 for b1, b2 in zip(bits_originais, bits_jpeg))
print(f"Taxa de erro pós-JPEG = {erros_jpeg/len(bits_originais):.4f} "
      f"({erros_jpeg}/{len(bits_originais)} bits)")
print("Mensagem pós-JPEG:", repr(bits_para_texto(bits_jpeg)))

# Decodificação da imagem (fluxo principal, sem ataques)

# 15) Carregamento da imagem esteganográfica e da chave

In [ ]:
X_estego_carregado = np.load("imagem_estego_U.npy")
dados_chave = np.load("chave_svd_U.npz")

U_original = dados_chave["U"]
s_original = dados_chave["s"]
VT_original = dados_chave["VT"]
n_bits = int(dados_chave["n_bits"])
theta = float(dados_chave["theta"])
inicio_colunas = int(dados_chave["inicio_colunas"])

plt.figure(figsize=(8, 4))
plt.imshow(X_estego_carregado, cmap="gray")
plt.title("Imagem Esteganográfica — Alterando U")
plt.axis("off")
plt.show()

# 16) Recuperação da matriz $Q$

$$U_{\text{estego}} = X_{\text{estego}} \, V \, S^{-1}, \qquad Q = U^T U_{\text{estego}}$$

In [ ]:
S_inv = np.diag(1 / s_original)
V_original = VT_original.T

U_estego_recuperado = X_estego_carregado @ V_original @ S_inv
Q_recuperado = U_original.T @ U_estego_recuperado

# 17) Extração dos bits e reconstrução da mensagem

Se a entrada $Q[j,i]$ (abaixo da diagonal do bloco de rotação) for positiva, o bit é 1; se for negativa, o bit é 0.

In [ ]:
bits_recuperados = ""
for idx in range(n_bits):
    i = inicio_colunas + 2 * idx
    j = i + 1
    bits_recuperados += "1" if Q_recuperado[j, i] > 0 else "0"

mensagem_recuperada = bits_para_texto(bits_recuperados)

print("Bits recuperados:")
print(bits_recuperados)
print("\nMensagem recuperada:")
print(mensagem_recuperada)

# 18) Avaliação: Taxa de Erro de Bits

Mesma métrica calculada no método de $\Sigma$, para a tabela comparativa de Resultados.

In [ ]:
L = len(bits_originais)
comparaveis = bits_recuperados[:L]
erros = sum(b1 != b2 for b1, b2 in zip(bits_originais, comparaveis))
taxa_erro = erros / L
print(f"Taxa de erro de bits = {taxa_erro:.4f} ({erros}/{L} bits incorretos)")

# Explicação e comparação com o método de Σ